# Setup: always run this.

In [1]:
import torch

#@torch._dynamo.config.patch(recompile_limit=2)
torch._dynamo.config.recompile_limit = 1
torch._dynamo.config.accumulated_recompile_limit = 10_000_000  # Defaults to 256.
torch._dynamo.config.fail_on_recompile_limit_hit = True

In [2]:
def core(x):
    return x.sum()

## Explicitly separate functions: works

In [3]:
@torch.compile(fullgraph=True, dynamic=False)
def frontendA(x, n):
    return core(x) + n

@torch.compile(fullgraph=True, dynamic=False)
def frontendB(x, n):
    return core(x) + n

frontendA(torch.ones(3), 3)  # Works, of course
frontendB(torch.ones(4), 3)  # Also works, nice, separate caches.

tensor(7.)

## Factory: doesn't work!

In [3]:
from functools import cache

@cache
def factory(key):
    @torch.compile(fullgraph=True, dynamic=False)
    def frontend(x, n):
        return core(x) + n
    return frontend

factory("foo")(torch.ones(3), 3)  # Works, of course
factory("bar")(torch.ones(3), 3)  # Works, ok...
factory("baz")(torch.ones(4), 3)  # COMPILE LIMIT!!

W1209 15:50:48.715000 2594107 /storage/home/qkv/.conda/envs/cenv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1550] [0/1] torch._dynamo hit config.recompile_limit (1)
W1209 15:50:48.715000 2594107 /storage/home/qkv/.conda/envs/cenv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1550] [0/1]    function: 'frontend' (/tmp/ipykernel_2594107/2898640909.py:5)
W1209 15:50:48.715000 2594107 /storage/home/qkv/.conda/envs/cenv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1550] [0/1]    last reason: 0/0: tensor 'x' size mismatch at index 0. expected 3, actual 4
W1209 15:50:48.715000 2594107 /storage/home/qkv/.conda/envs/cenv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1550] [0/1] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W1209 15:50:48.715000 2594107 /storage/home/qkv/.conda/envs/cenv/lib/python3.12/site-packages/torch/_dynamo/convert_frame.py:1550] [0/1] To diagnose recompilation issues, see https://pytorch.org/docs/

FailOnRecompileLimitHit: recompile_limit reached, because fail_on_recompile_limit_hit = True this is a HARD failure

## Code-cloning Factory: works

In [3]:
import types
from functools import cache

def clone_function(f):
    return types.FunctionType(f.__code__.replace(), f.__globals__, f.__name__, argdefs=f.__defaults__, closure=f.__closure__)

@cache
def factory(key):
    def frontend(x, n):
        return core(x) + n

    return torch.compile(clone_function(frontend), fullgraph=True, dynamic=False)

factory("foo")(torch.ones(3), 3)  # Works, of course
factory("bar")(torch.ones(3), 3)  # Works, ok...
factory("baz")(torch.ones(4), 3)  # Works too.

tensor(7.)